In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/usloth-docs-jsonl/usloth_docs.jsonl


In [2]:
#!pip install git+https://github.com/unslothai/unsloth.git

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-i610few4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-i610few4
  Resolved https://github.com/unslothai/unsloth.git to commit d78bf49045eee8a904f35cb1742841dcd723a626
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.6.12-py3-none-any.whl size=295104 sha256=affd706dd4bf4c285854b2d769bae1964c281c0e4994a2869507113c39d14344
  Stored in directory: /tmp/pip-ephem-wheel-cache-2_qay9ho/wheels/d1/17/05/850ab10c33284a4763b0595cd8ea9d01fce6e221cac24b3c01
Successfully built unsloth


In [5]:
!pip install  transformers datasets jsonlines  nltk tqdm --quiet

In [4]:
!pip install bitsandbytes --prefer-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 27.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 72.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found

In [7]:
pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 kB 4.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.0/129.0 kB 10.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
# === 🔧 IMPORTS ===
from unsloth import FastLanguageModel
from transformers import TrainingArguments, pipeline, AutoModelForCausalLM, AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from datasets import Dataset
import pandas as pd
import torch
from nltk.translate.bleu_score import sentence_bleu
from tqdm import tqdm
import os


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-07-01 10:04:49.685573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751364289.892626      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751364289.955896      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [9]:
# CHECK GPU 
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA Available: True
GPU: Tesla T4


In [10]:
# MODEL INFERENCE WRAPPER (Before & After Fine-Tuning) 
class BasicModelRunner:
    def __init__(self, model_name):
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.pipeline = pipeline("text-generation", model=self.model, tokenizer=self.tokenizer)

    def __call__(self, prompt, max_tokens=100):
        return self.pipeline(prompt, max_new_tokens=max_tokens)[0]['generated_text']


In [11]:
# INFERENCE BEFORE FINE-TUNING ===
print("\n=== 🔍 Non-finetuned output ===")
non_finetuned = BasicModelRunner("unsloth/llama-3-8b-bnb-4bit")
baseline_prompt = "### Question:\nTell me how to train my dog to sit\n\n### Answer:"
print(non_finetuned(baseline_prompt))



=== 🔍 Non-finetuned output ===


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Device set to use cuda:0


### Question:
Tell me how to train my dog to sit

### Answer: 
1. Start by teaching your dog to sit on command. Place your dog in a sit position, then say "Sit." 
2. To encourage your dog to stay in the sit position, you can give the command "Stay" and take a few steps back. 
3. Praise your dog for staying in the sit position, and reward them with a treat. 
4. Repeat this exercise several times a day, gradually increasing the distance you take from your dog. 
5.


In [26]:
#  LOAD INSTRUCTION DATASET
filename = "/kaggle/input/usloth-docs/usloth_docs.jsonl" 

import jsonlines
data = []
with jsonlines.open(filename) as reader:
    for obj in reader:
        data.append(obj)

instruction_dataset_df = pd.DataFrame(data)


In [27]:
# FORMAT PROMPTS
prompt_template_q = """### Question:\n{question}\n\n### Answer:"""
dataset = []

for i, row in instruction_dataset_df.iterrows():
    q = row["question"]
    a = row["answer"]
    formatted = prompt_template_q.format(question=q)
    dataset.append({"question": formatted, "answer": a})

hf_dataset = Dataset.from_list(dataset)


In [28]:
# LOAD MODEL WITH UNSLOTH 
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True
)


==((====))==  Unsloth 2025.6.12: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [29]:
# TOKENIZE DATA 
def formatting_prompts_func(example):
    return [f"{example['question']}\n{example['answer']}"]

tokenized_dataset = hf_dataset.map(
    lambda x: tokenizer(*formatting_prompts_func(x), truncation=True, padding="max_length", max_length=2048),
    remove_columns=hf_dataset.column_names,
    batched=False
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [30]:
#fine tunning
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "/kaggle/working/usloth-finetuned",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    warmup_steps = 5,
    max_steps = 20,
    learning_rate = 2e-4,
    logging_steps = 1,
    fp16 = True,
    save_strategy = "no",
)

# Pass model explicitly
trainer = FastLanguageModel.for_training(model)

# Manually set remaining fields
trainer.args = training_args
trainer.train_dataset = tokenized_dataset
trainer.data_collator = data_collator
trainer.tokenizer = tokenizer  # Good practice

# Train
trainer.train()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [31]:

model.save_pretrained("/kaggle/working/usloth-finetuned")
tokenizer.save_pretrained("/kaggle/working/usloth-finetuned")


('/kaggle/working/usloth-finetuned/tokenizer_config.json',
 '/kaggle/working/usloth-finetuned/special_tokens_map.json',
 '/kaggle/working/usloth-finetuned/tokenizer.json')

In [33]:
from unsloth import FastLanguageModel
from transformers import pipeline

# Load model (Unsloth handles device_map internally)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/kaggle/working/usloth-finetuned",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# ⚠️ DO NOT set `device=0` or any `device_map` in pipeline
# It must match the model's current device setup
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

# Test inference
prompt = "### Question:\nWhat is machine learning?\n\n### Answer:"
print(pipe(prompt, max_new_tokens=100)[0]["generated_text"])

==((====))==  Unsloth 2025.6.12: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# === 📊 BLEU SCORE EVALUATION ===
print("\n=== 📊 BLEU Evaluation ===")
total_score = 0

for i in tqdm(range(min(5, len(dataset)))):
    prompt = dataset[i]["question"]
    reference = [dataset[i]["answer"].split()]
    output = finetuned_model(prompt)
    candidate = output.replace(prompt, "").strip().split()
    score = sentence_bleu(reference, candidate)
    print(f"Sample {i+1} BLEU: {score:.4f}")
    total_score += score

print(f"\n🧮 Average BLEU Score: {total_score / 5:.4f}")
